# 5.1.4 Model 4: Support Vector Regression (SVR)

Kernel-based model (RBF kernel), a different modelling paradigm from the linear baseline and the two tree ensembles (Random Forest, XGBoost) already covered. Trained on `X_train_scaled.csv` (log1p-transformed numeric features, per Section 3.11.2) since SVR is scale-sensitive, same reasoning as the Linear Regression baseline.

In [1]:
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
import sys, os
sys.path.append(os.path.dirname(os.path.abspath('__file__')))
from model_utils import evaluate_model, cross_validate_model

MODELLING_DIR = os.path.join("..", "data", "modelling")

X_train = pd.read_csv(os.path.join(MODELLING_DIR, "X_train_scaled.csv"))
X_test = pd.read_csv(os.path.join(MODELLING_DIR, "X_test_scaled.csv"))
y_train = pd.read_csv(os.path.join(MODELLING_DIR, "y_train.csv"))["price"]
y_test = pd.read_csv(os.path.join(MODELLING_DIR, "y_test.csv"))["price"]

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

X_train: (3004, 51) | X_test: (751, 51)


## Hyperparameter Tuning
A small grid searched with 5-fold CV on the training set only (`GridSearchCV` scores on log(price) internally for ranking configurations, same convention as the Random Forest and XGBoost notebooks). `C` controls the regularisation strength, `epsilon` the width of the no-penalty margin around each prediction, and `gamma` the RBF kernel's reach.

In [2]:
param_grid = {
    "C": [1, 10, 100],
    "epsilon": [0.01, 0.1],
    "gamma": ["scale", 0.01],
}

grid_search = GridSearchCV(
    SVR(kernel="rbf"),
    param_grid,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print(f"Best CV score (log-scale RMSE): {-grid_search.best_score_:.4f}")

Best params: {'C': 10, 'epsilon': 0.1, 'gamma': 'scale'}
Best CV score (log-scale RMSE): 0.2659


## Train Final Model

In [3]:
model = SVR(kernel="rbf", **grid_search.best_params_)
model.fit(X_train, y_train)
print("Model trained.")

Model trained.


## Evaluate
Metrics computed on both train and test sets — the gap between them is needed for Section 6.2's overfitting/underfitting analysis.

In [4]:
train_metrics = evaluate_model(model, X_train, y_train, label="Train")
print()
test_metrics = evaluate_model(model, X_test, y_test, label="Test")

Train RMSE:  RM 142,922  (40.8% of median price)
Train MAE:   RM 71,544
Train MAPE:  17.7%
Train R2:    0.8099
Train MSE:   20,426,738,616

Test RMSE:  RM 186,687  (51.9% of median price)
Test MAE:   RM 85,716
Test MAPE:  17.9%
Test R2:    0.6834
Test MSE:   34,851,907,636


## Sanity check: permutation importance vs EDA (Section 4.5.2)
SVR with an RBF kernel has no `coef_`/`feature_importances_` (unlike the linear model's coefficients or the tree ensembles' impurity-based importances), so permutation importance is used instead: each feature is shuffled and the resulting drop in test R² is measured. Section 4.5.2 ranked Property Size, Bathroom, Parking Lot, and the Has_Gymnasium/Has_Swimming_Pool amenities as the strongest numerical correlates of price — this checks whether SVR's learned importances broadly agree.

In [5]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    model, X_test, y_test, scoring="r2", n_repeats=10, random_state=42, n_jobs=-1,
)
importance_table = pd.Series(perm_result.importances_mean, index=X_test.columns).sort_values(ascending=False)
print("Top 15 features by permutation importance (mean R2 drop):")
print(importance_table.head(15))

Top 15 features by permutation importance (mean R2 drop):
Property Size                     0.502531
State_Penang                      0.103779
PropertyType_Service_Residence    0.043029
PropertyType_Condominium          0.029250
State_Sabah                       0.025934
Bedroom                           0.024321
Property Age                      0.019678
PropertyType_Flat                 0.017451
State_Kuala_Lumpur                0.013694
State_Selangor                    0.012679
Parking Lot                       0.012372
Property_Age_Missing              0.012234
Bathroom                          0.010702
State_Unknown                     0.009518
Has_Minimart                      0.008392
dtype: float64


## 5-fold Cross-Validation
Run on X_train only (X_test stays untouched) to get a more robust performance estimate than a single train/test split, per Section 5.2's cross-validation requirement.

In [6]:
cv_results = cross_validate_model(
    SVR(kernel="rbf", **grid_search.best_params_),
    X_train, y_train, n_splits=5,
)

5-fold CV (mean +/- std):
  RMSE:  RM 165,041 +/- 20,141  (46.9% of median price)
  MAE:   RM 82,326 +/- 4,257
  MAPE:  20.3% +/- 1.2%
  R2:    0.7395 +/- 0.0412
  MSE:   27,644,236,179 +/- 7,215,629,283


## Save Trained Model
Saved for the Streamlit prototype (Section 8) to load directly, without retraining.

In [7]:
import joblib
MODEL_DIR = os.path.join("..", "models")
os.makedirs(MODEL_DIR, exist_ok=True)
model_path = os.path.join(MODEL_DIR, "svr_model.pkl")
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

Model saved to ..\models\svr_model.pkl


In [8]:
# Demo: reload the saved model and predict on an existing test row
import joblib
model = joblib.load(os.path.join(MODEL_DIR, "svr_model.pkl"))

sample = X_test.iloc[[0]]           # first row of X_test as a stand-in example
preds_log = model.predict(sample)
preds_rm = np.exp(preds_log)

print(f"Predicted price: RM {preds_rm[0]:,.0f}")
print(f"Actual price:    RM {np.exp(y_test.iloc[0]):,.0f}")

Predicted price: RM 342,703
Actual price:    RM 390,000


In [9]:
print("y_test head (log-price):", y_test.head())
print("y_test dtype:", y_test.dtype)
print("Expected actual price (RM):", np.exp(y_test.iloc[0]))

y_test head (log-price): 0    12.873902
1    12.100712
2    12.089539
3    12.793859
4    12.206073
Name: price, dtype: float64
y_test dtype: float64
Expected actual price (RM): 389999.99999999924
